In [3]:
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import os
import requests
PASSWORD = os.getenv('SNOWSQL_PWD')
print(PASSWORD)

5eWxWv4EvyCkhkY


In [67]:
try:
    ctx = snowflake.connector.connect(
        user='ABHINAVSHARMA2002',
        password=PASSWORD,
        account='qraojwa-yg67137'
    )
    cs = ctx.cursor()
    try:
        cs.execute("CREATE WAREHOUSE IF NOT EXISTS task_1_warehouse_mg")
        cs.execute("CREATE DATABASE IF NOT EXISTS testdb_mg")
        cs.execute("USE DATABASE testdb_mg")
        cs.execute("CREATE SCHEMA IF NOT EXISTS task_2_mg")
        cs.execute("USE WAREHOUSE task_1_warehouse_mg")
        cs.execute("USE SCHEMA task_2_mg")
        ##cs.execute('DROP VIEW "product_f";')
        ##cs.execute('SELECT COUNT(*) AS invalid_dates FROM orders o WHERE o."date" <= 0')
        ##print(cs.fetchall())
        runQueries(cs)
        ##cs.execute('SELECT * FROM "v_cust_purch_summary_f" LIMIT 10')
        print(cs.fetchall())
    except Exception as e:
        print(f"Error: {e}") 
    finally:
        cs.close()
        ctx.close()
except Exception as e:
    print(f"Error: {e}")

[('View "product_f" successfully created.',)]


In [65]:
def runQueries(cs):    
    query = """
    CREATE OR REPLACE VIEW "product_f" AS 
WITH monthly_sales AS (
    SELECT 
        p."product_id", 
        p."product_name", 
        TO_CHAR(DATE_TRUNC('MONTH', TO_TIMESTAMP(o."date")),'MM') AS "calendar_month",
        SUM(o."quantity") AS "total_quantity_sold" 
    FROM products p
    JOIN orders o 
        ON p."product_id" = o."product_id" 
    WHERE o."date" > 0 -- Ensure valid dates
    GROUP BY p."product_id", p."product_name", "calendar_month", o."date"
), 
moving_avg AS (
    SELECT 
        "product_id", 
        "product_name",
        "calendar_month", 
        "total_quantity_sold", 
        AVG("total_quantity_sold") OVER (
            PARTITION BY "product_id" 
            ORDER BY "calendar_month" 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS "moving_3_month_avg_sales"  -- 3-month moving average
    FROM monthly_sales
)
SELECT * FROM moving_avg;

    """
    cs.execute(query)